# Covariance matrix estimation: methods compared

This notebook estimates the N x N covariance of daily returns with several methods and
compares them on the same data. With N names and T observations, the sample covariance
has N(N+1)/2 parameters. When N is near T or larger, the sample covariance is
ill-conditioned. Its largest eigenvalues are too large and its smallest eigenvalues are
too small. A minimum-variance optimiser then puts weight on the noisiest directions. Each
method below reduces this estimation error in a different way.

Methods: sample, Ledoit-Wolf linear shrinkage, OAS, Ledoit-Wolf nonlinear shrinkage
(QIS), a PCA statistical factor model, random-matrix eigenvalue clipping, EWMA
(RiskMetrics), and a GARCH conditional covariance with constant conditional correlation
(CCC). The last section shows DCC, which lets the correlation change with time.

Most of these methods keep the sample eigenvectors and change only the eigenvalues. The
middle section shows each method as a curve that maps a sample eigenvalue to a cleaned
eigenvalue. The final test is economic. Each estimate gives a global minimum-variance
(GMV) portfolio out of sample, and the score is the realised volatility of that
portfolio. A lower volatility is better.

In [ ]:
import warnings

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from arch import arch_model
from arch.utility.exceptions import ConvergenceWarning
from joblib import Parallel, delayed
from scipy.optimize import minimize
from sklearn.covariance import OAS, LedoitWolf

from sdp import dal
from sdp.risk import eligible, gmv, pca_cov, qis, rmt_cov, sample_cov

warnings.filterwarnings("ignore", category=ConvergenceWarning)
wh = dal.warehouse()

## The return panel

The panel holds liquid common stock only. `in_universe` excludes ETFs, because an ETF is
a linear combination of its constituents and lets the optimiser build near-riskless
pairs. The panel loads each name that is in the universe on one or more dates of the
window. A name has NaN on each session with no price.

The selection of names is point in time. At each rebalance date, a name is eligible when
it is in the universe on that date and has a return on each session of the estimation
window. The estimate uses the `N_NAMES` eligible names with the largest trailing dollar
volume on that date. A held name with no return on a session of the hold period
contributes a zero return on that session. This rule is an approximation. For a name that
leaves the tape, the weight earns zero from that session, and the backtest does not see a
delisting return. For a name with a gap in its prices, the backtest does not see the move
across the gap. The panel keys on `security_key`, so a ticker change stays one series. The
rule does not carry a price over a gap.

The structure sections use one matrix: the names that are eligible on the last date, with
the full panel as the estimation window. Thus each estimator in those sections gets the
same T x N matrix.

In [ ]:
N_NAMES = 2000     # names in each estimate
TOTAL = 780        # trailing sessions of prices

panel = wh.sql(f"""
    with win as (
        select distinct date from main_staging.stg_universe
        order by date desc limit {TOTAL}
    ), names as (
        select distinct security_key from main_staging.stg_universe
        where in_universe and date >= (select min(date) from win)
    )
    select p.date, u.security_key, p.adj_close_total, u.in_universe, u.adv
    from main_staging.stg_prices_adjusted p
    join main_staging.stg_universe u using (ticker, date)
    join names using (security_key)
    where p.date >= (select min(date) from win)
      -- One row per security and session, as in the marts.
      and u.is_primary_line
""").pl()


def wide(col):
    """Pivot one panel column to a sessions x securities frame, with the keys in order."""
    return (panel.pivot(on="security_key", index="date", values=col, sort_columns=True)
            .sort("date"))


px_wide = wide("adj_close_total")
rdates = px_wide["date"].to_numpy()[1:]                           # date of each return row
R_all = np.diff(np.log(px_wide.drop("date").to_numpy()), axis=0)  # NaN where no price
U_all = wide("in_universe").drop("date").fill_null(False).to_numpy()[1:]
A_all = wide("adv").drop("date").fill_null(0.0).to_numpy()[1:]

T = R_all.shape[0]
R = R_all[:, eligible(R_all, U_all, A_all, T, T, N_NAMES)]   # structure matrix, T x N
N = R.shape[1]
print(f"panel: {T} sessions, {R_all.shape[1]} names in the universe on one or more dates")
print(f"structure matrix: T={T} sessions, N={N} names, q=N/T={N / T:.2f}")
print(f"mean single-name vol {R.std(0).mean() * np.sqrt(252) * 100:.1f}%/yr, "
      f"equal-weight {R.mean(1).std() * np.sqrt(252) * 100:.1f}%/yr")

## The estimators

Each estimator takes a T x N return matrix and returns an N x N covariance. `sdp.risk`
holds the sample, QIS, PCA and RMT estimators and the GMV weights. Linear shrinkage and
OAS come from scikit-learn. This notebook builds EWMA and the GARCH model, so that their
mechanics are visible.

In [ ]:
def c_lw(X):
    return LedoitWolf().fit(X).covariance_        # linear shrinkage to a scaled identity


def c_oas(X):
    return OAS().fit(X).covariance_               # oracle-approximating shrinkage


def c_ewma(X, lam=0.94):
    """RiskMetrics exponentially weighted covariance. It has about 1/(1-lam) effective
    observations, so it is badly conditioned for a large N and a poor input to an inverse."""
    Xc = X - X.mean(0)
    t = len(Xc)
    w = (1 - lam) * lam ** np.arange(t)[::-1]
    w /= w.sum()
    return (Xc * w[:, None]).T @ Xc


def _fit_col(col):
    """Fit a univariate GARCH(1,1) and return the standardised return series."""
    try:
        res = arch_model(col * 100, mean="Zero", vol="GARCH", p=1, q=1,
                         rescale=False).fit(disp="off", show_warning=False)
        return col / (res.conditional_volatility / 100)
    except Exception:
        return col / col.std()


def c_ccc_garch(X):
    """Constant conditional correlation: the current GARCH volatilities on the diagonal and
    a shrunk correlation of the standardised residuals off the diagonal."""
    def _fit_both(col):
        try:
            res = arch_model(col * 100, mean="Zero", vol="GARCH", p=1, q=1,
                             rescale=False).fit(disp="off", show_warning=False)
            s = res.conditional_volatility / 100
            return s[-1], col / s
        except Exception:
            sd = col.std()
            return sd, col / sd
    out = Parallel(n_jobs=-1)(delayed(_fit_both)(X[:, j]) for j in range(X.shape[1]))
    d_now = np.array([o[0] for o in out])
    Z = np.column_stack([o[1] for o in out])
    Rz = LedoitWolf().fit(Z).covariance_
    dz = np.sqrt(np.diag(Rz))
    Rz = Rz / np.outer(dz, dz)
    D = np.diag(d_now)
    return D @ Rz @ D


ESTIMATORS = {"sample": sample_cov, "ledoit_wolf": c_lw, "oas": c_oas, "nonlin_lw": qis,
              "pca": pca_cov, "rmt": rmt_cov, "ewma": c_ewma, "ccc_garch": c_ccc_garch}
FAST = {k: v for k, v in ESTIMATORS.items() if k != "ccc_garch"}

## Structure of each estimate

The table gives the condition number of each estimate on the structure matrix. A large
condition number makes the inverse, and thus the GMV weights, sensitive to noise.

In [ ]:
rows = []
for name, fn in FAST.items():
    S = fn(R)
    rows.append({"method": name, "condition_number": float(np.linalg.cond(S))})
pl.DataFrame(rows).sort("condition_number")

In [ ]:
# Eigenvalue spectra on a log scale. The sample spectrum has a few large eigenvalues and a
# long tail of small ones. When N>T, the tail is near zero and falls off the log axis. The
# regularised methods lift the floor. QIS keeps the top and lifts the bottom.
fig = go.Figure()
for name in ("sample", "ledoit_wolf", "oas", "nonlin_lw", "pca", "rmt"):
    ev = np.sort(np.linalg.eigvalsh(ESTIMATORS[name](R)))[::-1]
    fig.add_trace(go.Scatter(y=ev, mode="lines", name=name))
fig.update_yaxes(type="log", title="eigenvalue (log)")
fig.update_layout(title="Covariance eigenvalue spectra (structure matrix)",
                  xaxis_title="rank", height=440)
fig.show()

## What the estimators do to the eigenvalues

The sample, both linear shrinkages and RMT clipping keep the sample eigenvectors and
change only the eigenvalues. Each method is then a curve that maps a sample eigenvalue to
a cleaned eigenvalue. The Marchenko-Pastur result below shows why a curve is necessary.
Pure noise spreads the eigenvalues into a band, so the largest sample eigenvalues are
biased up and the smallest sample eigenvalues are biased down.

- **Linear shrinkage (LW, OAS)** is a straight line. It pulls each eigenvalue toward the
  mean by one intensity. The legend gives the intensity of each.
- **Nonlinear shrinkage (QIS)** is a curve. It keeps the top (signal) almost unchanged and
  lifts the bottom (noise) toward a floor. Among the estimators that keep the sample
  eigenvectors, it is asymptotically optimal.
- **RMT clipping** is a step. It keeps the eigenvalues above the noise edge and sets the
  whole bulk to one level.

In [ ]:
# Marchenko-Pastur: the band where pure noise puts the eigenvalues. Correlation eigenvalues
# inside [lam_minus, lam_plus] agree with noise. Eigenvalues above the edge are signal
# (real common factors). Thus one flat shrinkage is wrong and a curve is necessary.
C = np.corrcoef(R, rowvar=False)
evc = np.sort(np.linalg.eigvalsh(C))[::-1]
qf = N / T
lam_m, lam_p = (1 - np.sqrt(qf)) ** 2, (1 + np.sqrt(qf)) ** 2
xs = np.linspace(max(lam_m, 1e-6), lam_p, 300)
mp = np.sqrt(np.clip((lam_p - xs) * (xs - lam_m), 0, None)) / (2 * np.pi * qf * xs)
n_sig = int((evc > lam_p).sum())
n_null = int((evc <= 1e-8).sum())
bulk = evc[(evc > 1e-8) & (evc <= lam_p * 1.5)]       # nonzero, near the band

fig = go.Figure()
fig.add_trace(go.Histogram(x=bulk, histnorm="probability density", nbinsx=80,
                           name="sample eigenvalues", marker_color="#8b949e"))
fig.add_trace(go.Scatter(x=xs, y=mp, mode="lines", name="Marchenko-Pastur (pure noise)",
                         line=dict(color="#cf222e", width=2)))
fig.add_vline(x=lam_p, line_dash="dash", line_color="#1a7f37",
              annotation_text=f"edge {lam_p:.2f}")
fig.update_layout(title=f"Correlation eigenvalues vs the MP noise band "
                        f"({n_sig} above the edge = signal, top = {evc[0]:.0f})",
                  xaxis_title="eigenvalue", yaxis_title="density",
                  xaxis_range=[0, lam_p * 1.5], height=420)
fig.show()
print(f"q=N/T={qf:.2f}   noise band [{lam_m:.2f}, {lam_p:.2f}]   "
      f"{n_sig} of {N} eigenvalues are signal, {n_null} are null, "
      f"market factor = {evc[0]:.0f}")

In [ ]:
# The shrinkage curve maps a sample eigenvalue to a cleaned eigenvalue, for the methods
# that keep the sample eigenvectors. The sample is the 45-degree line. Linear shrinkage is
# a straight pull toward the mean. QIS is a curve that keeps the top and lifts the bottom.
# RMT clipping is a step at the noise edge.
S = np.cov(R, rowvar=False)
lam = np.sort(np.linalg.eigvalsh(S))                  # ascending
mu = lam.mean()
s_lw = LedoitWolf().fit(R).shrinkage_
s_oas = OAS().fit(R).shrinkage_
lin = (1 - s_lw) * lam + s_lw * mu
oas = (1 - s_oas) * lam + s_oas * mu
lam_nl, d_nl = qis(R, return_spectrum=True)
# The RMT clip in covariance space, so that it is a function of the same lam.
edge = mu * (1 + np.sqrt(N / T)) ** 2
keep = lam > edge
step = np.where(keep, lam, lam[~keep].mean() if (~keep).any() else mu)
step *= lam.sum() / step.sum()

pos = lam > 0                                         # log axis: drop the null space
fig = go.Figure()
fig.add_trace(go.Scatter(x=lam[pos], y=lam[pos], mode="lines", name="sample (no change)",
                         line=dict(color="#8b949e", dash="dash")))
fig.add_trace(go.Scatter(x=lam[pos], y=lin[pos], mode="lines",
                         name=f"linear LW (delta={s_lw:.2f})", line=dict(color="#1f6feb")))
fig.add_trace(go.Scatter(x=lam[pos], y=oas[pos], mode="lines",
                         name=f"OAS (delta={s_oas:.2f})", line=dict(color="#8250df")))
fig.add_trace(go.Scatter(x=lam_nl[lam_nl > 0], y=d_nl[lam_nl > 0], mode="lines",
                         name="nonlinear LW (QIS)", line=dict(color="#cf222e", width=3)))
fig.add_trace(go.Scatter(x=lam[pos], y=step[pos], mode="lines", name="RMT clip",
                         line=dict(color="#1a7f37", shape="hv")))
fig.add_vline(x=edge, line_dash="dot", line_color="#1a7f37", annotation_text="MP edge")
fig.update_xaxes(type="log", title="sample eigenvalue (log)")
fig.update_yaxes(type="log", title="cleaned eigenvalue (log)")
fig.update_layout(title="What each method does to a sample eigenvalue", height=520)
fig.show()

In [ ]:
# QIS on the structure matrix and on the names eligible over the last 126 sessions. The
# table compares the smallest eigenvalue and the condition number with the sample matrix.
def _regime(Xw, label):
    Tw, Nw = Xw.shape
    Snl = qis(Xw)
    Ssa = sample_cov(Xw)
    ev = np.linalg.eigvalsh(Snl)
    return {"regime": label, "T": Tw, "N": Nw, "q_NoverT": round(Nw / Tw, 2),
            "nls_min_eig": float(ev.min()), "nls_posdef": bool(ev.min() > 0),
            "nls_cond": round(float(np.linalg.cond(Snl)), 1),
            "sample_cond": float(np.linalg.cond(Ssa))}


R_126 = R_all[T - 126:, eligible(R_all, U_all, A_all, T, 126, N_NAMES)]
pl.DataFrame([_regime(R, "full sample"), _regime(R_126, "last 126 sessions")])

### One estimator, three constructions

The shrinkage curve above is one object, the Ledoit-Peche oracle:

`d(lam) = lam / |1 - c + c*lam*g(lam + i0)|^2`, with `c = N/T` and
`g(z) = (1/N) sum_j 1/(z - lam_j)`, the Stieltjes transform of the sample spectrum.

Three constructions give this oracle:

- **QIS** (above) writes the oracle in inverse-eigenvalue space. It works in both regimes,
  with no tuning and no special functions.
- **Epanechnikov kernel** (Ledoit-Wolf 2020) estimates the spectral density and its
  Hilbert transform. The Hilbert transform of this kernel has a closed form (a log), from
  a principal-value integral.
- **Stieltjes / Cauchy** evaluates `g` a small distance above the real axis. By
  Sokhotski-Plemelj, the density and its Hilbert transform are the imaginary and real
  parts of that one complex sum. Thus this construction computes no Hilbert transform.
  The small imaginary part is the bandwidth.

**T>N and N>T are different problems.** When N>T, the sample has N - n null eigenvalues
(n = T - 1 after demeaning), with probability mass `N_null/N` at zero. That mass must
enter `g`. Without it, the non-null eigenvalues shrink too far. In the Cauchy form, the
mass is one extra term (`nz / z`), so that construction stays correct when N/T crosses 1.
The Epanechnikov kernel needs the same correction and also a good density estimate near
zero, which it does not have. Thus it moves away from QIS when N>T. QIS avoids both
problems because it works in inverse-eigenvalue space. The cell below prints the
agreement for the current `q = N/T`.

All three constructions cost `O(N^2)` for the sum over eigenvalue pairs. For N in the
thousands, the sum takes milliseconds. For a larger N, a fast multipole method or an FFT
on a binned density makes the sum nearly linear.

In [ ]:
def _pos_eig(X):
    """Return the sample eigenvalues and eigenvectors, the non-null mask, n and c = N/n."""
    T, N = X.shape
    Xc = X - X.mean(0)
    n = T - 1
    Sm = (Xc.T @ Xc) / n
    Sm = (Sm + Sm.T) / 2
    lam, u = np.linalg.eigh(Sm)
    lam = np.clip(lam, 0, None)
    keep = lam > lam.max() * N * np.finfo(lam.dtype).eps
    return lam, u, keep, n, N / n


def nls_stieltjes(X, eta_scale=0.5):
    """Cauchy construction: evaluate g a small distance above the real axis. The density
    and its Hilbert transform are the imaginary and real parts of one complex sum, and the
    imaginary part of z is the bandwidth. The `nz / z` term puts the null mass of an N>T
    sample at zero, so this form is correct in both regimes."""
    lam, u, keep, n, c = _pos_eig(X)
    N = lam.size
    lk = lam[keep]
    nz = N - lk.size
    h = (min(c ** 2, 1 / c ** 2) ** 0.35) / N ** 0.35
    z = lk + 1j * eta_scale * h * lk
    g = (np.sum(1.0 / (z[:, None] - lk[None, :]), axis=1) + nz / z) / N   # null mass at 0
    d = np.empty(N)
    d[keep] = lk / np.abs(1 - c + c * lk * g) ** 2
    d[~keep] = 1.0 / ((c - 1) * np.mean(1 / lk)) if c > 1 else lk.min()
    return lam, d * (lam.sum() / d.sum())


r5 = np.sqrt(5.0)


def _Hk(x):
    """Hilbert transform of the Epanechnikov kernel, a closed form from a principal-value
    integral: H k(x) = 3x/(10 pi) + 3/(4 sqrt5 pi) (1 - x^2/5) log|(sqrt5 + x)/(sqrt5 - x)|."""
    with np.errstate(divide="ignore", invalid="ignore"):
        out = (3 * x / (10 * np.pi)
               + 3 / (4 * r5 * np.pi) * (1 - x ** 2 / 5) * np.log(np.abs((r5 + x) / (r5 - x))))
    return np.where(np.abs(x) == r5, 3 * x / (10 * np.pi), out)


def nls_epanechnikov(X):
    """Ledoit-Wolf 2020 analytical form: a variable-bandwidth Epanechnikov estimate of the
    spectral density f and of its Hilbert transform, in the same oracle. The null mass
    enters the Hilbert part as N_null/(N*lam). For N>T the kernel density near zero is
    poor, so the result moves away from QIS."""
    lam, u, keep, n, c = _pos_eig(X)
    N = lam.size
    lk = lam[keep]
    nz = N - lk.size
    hb = n ** (-1 / 3) * lk[None, :]                    # local bandwidth lam_j * n^-1/3
    xg = (lk[:, None] - lk[None, :]) / hb
    f = np.mean(0.75 / r5 * np.maximum(1 - xg ** 2 / 5, 0) / hb, axis=1)
    G = np.mean(np.pi * _Hk(xg) / hb, axis=1) + (nz / N) / lk   # Hilbert transform + null mass
    g = G - 1j * np.pi * f
    d = np.empty(N)
    d[keep] = lk / np.abs(1 - c + c * lk * g) ** 2
    d[~keep] = 1.0 / ((c - 1) * np.mean(1 / lk)) if c > 1 else lk.min()
    return lam, d * (lam.sum() / d.sum())

In [ ]:
# The plot overlays the three curves on the structure matrix. The agreement uses the
# non-null eigenvalues only, because the sample does not see the null space.
lam_q, d_q = qis(R, return_spectrum=True)
lam_s, d_s = nls_stieltjes(R)
lam_e, d_e = nls_epanechnikov(R)
p = lam_q > lam_q.max() * len(lam_q) * np.finfo(lam_q.dtype).eps    # non-null eigenvalues
fig = go.Figure()
fig.add_trace(go.Scatter(x=lam_q[p], y=lam_q[p], mode="lines", name="sample (no change)",
                         line=dict(color="#8b949e", dash="dash")))
fig.add_trace(go.Scatter(x=lam_q[p], y=d_q[p], mode="lines", name="QIS (inverse algebra)",
                         line=dict(color="#cf222e", width=3)))
fig.add_trace(go.Scatter(x=lam_e[p], y=d_e[p], mode="markers", name="Epanechnikov kernel",
                         marker=dict(color="#1f6feb", size=5)))
fig.add_trace(go.Scatter(x=lam_s[p], y=d_s[p], mode="markers", name="Stieltjes (Cauchy)",
                         marker=dict(color="#1a7f37", size=5, symbol="x")))
fig.update_xaxes(type="log", title="sample eigenvalue (log)")
fig.update_yaxes(type="log", title="cleaned eigenvalue (log)")
fig.update_layout(title=f"One oracle, three constructions (q=N/T={N / T:.2f})", height=480)
fig.show()


def med(d):
    return float(np.median(np.abs(d[p] - d_q[p]) / d_q[p]))


print(f"regime {'T>N' if N <= T else 'N>T'}")
print(f"median |rel diff| vs QIS on non-null eigenvalues:  "
      f"Stieltjes {med(d_s):.1%}   Epanechnikov {med(d_e):.1%}")

## Economic comparison: out-of-sample minimum-variance risk

At each rebalance date, the backtest selects the eligible names (see the return panel),
estimates the covariance on the trailing window, forms the GMV portfolio and holds it to
the next rebalance. The score is the realised volatility of the joined out-of-sample
returns. The gross leverage (the sum of the absolute weights) shows how each estimator
behaves. A short `EST_WIN` makes q = N/EST_WIN larger and puts more stress on the
estimators.

In [ ]:
def backtest(fn, win, step=21):
    """Return the annualised out-of-sample GMV volatility in percent, the mean gross
    leverage and the mean count of names, over the rebalances."""
    oos, lev, size = [], [], []
    for d in range(win, T - 1, step):
        cols = eligible(R_all, U_all, A_all, d, win, N_NAMES)
        w = gmv(fn(R_all[d - win:d, cols]))
        lev.append(float(np.abs(w).sum()))
        size.append(cols.size)
        oos.append(np.nan_to_num(R_all[d:d + step, cols]) @ w)   # no return: zero
    o = np.concatenate(oos)
    return o.std() * np.sqrt(252) * 100, float(np.mean(lev)), float(np.mean(size))


EST_WIN, STEP = 126, 21    # 126 sessions make q = N/win large enough to separate the methods

results = []
for name, fn in ESTIMATORS.items():      # ccc_garch fits GARCH again in each window (slow)
    vol, lev, n = backtest(fn, EST_WIN, STEP)
    results.append({"method": name, "oos_vol_pct": round(vol, 2),
                    "gross_leverage": round(lev, 1)})
res = pl.DataFrame(results).sort("oos_vol_pct")
print(f"EST_WIN={EST_WIN}  mean N={n:.0f}  q=N/win={n / EST_WIN:.1f}")
res

In [ ]:
fig = px.bar(res.to_pandas(), x="method", y="oos_vol_pct", color="gross_leverage",
             color_continuous_scale="Reds",
             title=f"Out-of-sample GMV volatility (EST_WIN={EST_WIN})")
fig.update_layout(yaxis_title="annualised vol %", xaxis_title="", height=440)
fig.show()

In [ ]:
# The regime effect: a shorter estimation window gives a larger q. The table gives the
# out-of-sample volatility of each fast method for three windows.
grid = []
for win in (252, 126, 90):
    for name, fn in FAST.items():
        vol, _, n = backtest(fn, win, STEP)
        grid.append({"est_win": win, "method": name, "oos_vol_pct": round(vol, 2)})
    print(f"EST_WIN={win}  mean N={n:.0f}  q=N/win={n / win:.1f}")
pl.DataFrame(grid).pivot(values="oos_vol_pct", index="method", on="est_win").sort("method")

## DCC: time-varying correlation

The CCC model above holds the correlation matrix constant. DCC (Engle) lets the
correlation change: the correlation of the standardised residuals follows its own scalar
recursion. The notebook estimates DCC by quasi-maximum likelihood on the `N_DCC` names
with the largest dollar volume in the structure matrix. The likelihood inverts an N x N
matrix at each step, so the cost increases quickly with N. The plot shows the mean
pairwise correlation over time. A constant-correlation model holds this line flat.

In [ ]:
N_DCC = min(60, N)
Zc = np.column_stack(Parallel(n_jobs=-1)(delayed(_fit_col)(R[:, j]) for j in range(N_DCC)))
Zc = Zc / Zc.std(0)
Qbar = LedoitWolf().fit(Zc).covariance_
dq = np.sqrt(np.diag(Qbar))
Qbar = Qbar / np.outer(dq, dq)
iu = np.triu_indices(N_DCC, 1)


def dcc_nll(p):
    """Return the DCC negative log-likelihood, without its constant, for p = (a, b)."""
    a, b = p
    if a <= 0 or b <= 0 or a + b >= 0.999:
        return 1e12
    Q = Qbar.copy()
    nll = 0.0
    for t in range(len(Zc)):
        dd = np.sqrt(np.diag(Q))
        Rt = Q / np.outer(dd, dd)
        _, logdet = np.linalg.slogdet(Rt)
        nll += logdet + Zc[t] @ np.linalg.solve(Rt, Zc[t])
        z = Zc[t]
        Q = (1 - a - b) * Qbar + a * np.outer(z, z) + b * Q
    return nll


fit = minimize(dcc_nll, [0.02, 0.95], method="Nelder-Mead",
               options={"xatol": 1e-3, "fatol": 1e-1, "maxiter": 60})
a, b = fit.x
print(f"DCC a={a:.3f}  b={b:.3f}  persistence={a + b:.3f}  (N={N_DCC})")

Q = Qbar.copy()
avg_corr = []
for t in range(len(Zc)):
    dd = np.sqrt(np.diag(Q))
    Rt = Q / np.outer(dd, dd)
    avg_corr.append(Rt[iu].mean())
    z = Zc[t]
    Q = (1 - a - b) * Qbar + a * np.outer(z, z) + b * Q

fig = go.Figure()
fig.add_trace(go.Scatter(x=rdates, y=avg_corr, mode="lines", name="DCC (dynamic)",
                         line=dict(color="#cf222e", width=1)))
fig.add_hline(y=float(np.mean(avg_corr)), line_dash="dash", line_color="#1f6feb",
              annotation_text="constant-correlation level")
fig.update_layout(title=f"Average pairwise correlation, {N_DCC} names",
                  yaxis_title="mean correlation", height=400)
fig.show()

## Takeaways

- **Sample** is usable only when T is much larger than N. When N is near or above the
  window length, its GMV portfolio has a very large gross leverage and an out-of-sample
  risk far above that of the other methods.
- **Linear shrinkage (LW, OAS)** is a good statistical default. It needs no factor
  assumption, it is well conditioned, and its out-of-sample risk is far below that of the
  sample. It uses one intensity for the whole spectrum, so its shrinkage curve is a
  straight line.
- **Nonlinear shrinkage (QIS)** gives each eigenvalue its own correction. The shrinkage
  curve shows the mechanism: QIS keeps the signal eigenvalues and lifts only the noise. It
  stays positive definite when N>T. In these backtests its out-of-sample risk is close to
  that of linear shrinkage.
- **PCA and RMT clipping** use the factor structure of the returns. At the shorter
  windows, their out-of-sample risk is lower than that of the shrinkage methods. RMT
  clipping is the correlation view of the same idea.
- **EWMA** is a reactive tool to measure risk, not an input for an optimiser at this N.
  Its condition number is the largest in the table, so its GMV portfolio is unstable.
- **CCC and DCC GARCH** give a conditional covariance from univariate GARCH models. DCC
  also lets the correlation change with time. In the backtest above, CCC has the lowest
  out-of-sample risk, at the cost of one GARCH fit for each name in each window. Its
  weights can favour a name with a low current volatility, for example a merger target
  near its deal price. Thus this result depends on the rule for a name that leaves the
  tape.

The platform has no characteristic data yet, so this notebook does not build a
fundamental (Barra-style) factor model. PCA is its statistical stand-in until the
ticker-details data is available.